In [ ]:
import yaml

with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

from types import SimpleNamespace

def dict_to_ns(d):
    return SimpleNamespace(**{
        k: dict_to_ns(v) if isinstance(v, dict) else v
        for k, v in d.items()
    })
cfg = dict_to_ns(cfg)
cfg

{'experiment': {'name': 'fc_hopping', 'seed': 42},
 'tx': {'ntx': 2,
  'power_dbm': 51,
  'pos': [[-290, 50, 10], [52, 142, 10]],
  'look_at_pos': [[-280, 100, 1], [52, 150, 1]],
  'pattern': ['tr38901', 'tr38901'],
  'polarization': ['V', 'V']},
 'ue': {'nue': 1,
  'pos': [[-290, 50, 10]],
  'orientation': [[0, 0, 0]],
  'min_speed': 0,
  'max_speed': 2,
  'nrx': 4,
  'rx_id': [0, 1, 2, 3],
  'rx_loc': ['top_left', 'bottom_right', 'center_left', 'center_right'],
  'rx_orientation': [[0, 0, 0], [0, 0, 0], [0, 0, 0], [0, 0, 0]]},
 'dataset_path': {'raw': '/dataset_multi_5mps/dataset_raw'}}

In [2]:
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU available:", gpus)
else:
    print("No GPU, using CPU")

print()


import os # Configure which GPU
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


from sionna.rt import load_scene, PlanarArray, Transmitter, Receiver, Camera,\
                      PathSolver, ITURadioMaterial, SceneObject

import matplotlib.pyplot as plt

import mitsuba as mi

import numpy as np

# Import Sionna utils from the wireless class
try:
    import sionnautils
except ImportError as e:
    # Install Sionna if package is not already installed
    !pip install git+https://github.com/sdrangan/wirelesscomm.git
    import sionnautils
    
    
    
from scipy.spatial.transform import Rotation as R

import mitsuba as mi
import drjit as dr
from sionna.rt import AntennaPattern, PlanarArray, register_antenna_pattern
from sionnautils.custom_scene import list_scenes, get_scene


2026-01-12 17:28:07.522405: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-12 17:28:07.570579: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-12 17:28:08.886885: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]



In [3]:
from sionnautils.custom_scene import list_scenes, get_scene
scenes = list_scenes()
print(scenes)

scene_path, map_data = get_scene('nyu_tandon')
for k, v in map_data.items():
    print(f'{k}: {v}')

scene = load_scene(scene_path,merge_shapes=True)

floor = scene.get('ground')
# print(f'Floor material: {floor.radio_material.name}')
floor.radio_material = ITURadioMaterial("itu_concrete",
                                "concrete",
                                thickness=0.01,
                                color=(0.5, 0.5, 0.5))

scene.remove("itu_wet_ground")

for name, obj in scene.objects.items():
    print(f'{name:<15}{obj.radio_material.name}')
# scene.render(camera=my_cam, num_samples=512)

scene.radio_materials

['nyu_tandon']
bbox_lat: [40.69012764197041, 40.699120858029595]
bbox_long: [-73.99156687083165, -73.97970552916836]
address: 5 MetroTech Center, Brooklyn, NY 11201
descr: NYU Tandon campus
2026-01-12 17:28:12 WARN  [HDRFilm] Monochrome mode enabled, setting film output pixel format to 'luminance' (was rgb).
no-name-1      itu_marble
ground         itu_concrete


{'itu_marble': ITURadioMaterial type=marble
                  eta_r=7.074
                  sigma=0.018
                  thickness=0.100
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000,
 'itu_concrete': ITURadioMaterial type=concrete
                  eta_r=5.240
                  sigma=0.123
                  thickness=0.010
                  scattering_coefficient=0.000
                  xpd_coefficient=0.000}

In [4]:
for index_tx in range(cfg.tx.ntx):
    
    scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern=cfg.tx.pattern[index_tx],
                             polarization=cfg.tx.polarization[index_tx])

    tx = Transmitter(name="tx-" + str(index_tx),
                            position=cfg.tx.pos[index_tx],
                            display_radius=2)

    # Add transmitter instance to scene
    scene.remove("tx-" + str(index_tx))
    scene.add(tx)

    tx.look_at(cfg.tx.look_at_pos[index_tx])

In [6]:
scene.preview()

In [ ]:
from collections import defaultdict

rx_fc = cfg["ue"]["rx_fc"]   # [1.5e10, 1.5e10, 3.5e9, 3.5e9]

fc_to_rx_idx = defaultdict(list)
for rx_idx, fc in enumerate(rx_fc):
    fc_to_rx_idx[fc].append(rx_idx)

n_unique_fc = len(fc_to_rx_idx)

print("Number of unique fc:", n_unique_fc)
print("fc -> rx indices:", dict(fc_to_rx_idx))
